<a href="https://colab.research.google.com/github/Somtochukwu438/quizgen-llama3/blob/main/notebooks/04_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip -q install transformers peft bitsandbytes accelerate chromadb sentence-transformers gradio beautifulsoup4
from google.colab import userdata, drive
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os, sys, json
REPO = "/content/quizgen-llama3"
if os.path.exists(REPO):
    !git -C {REPO} pull
else:
    !git clone https://github.com/Somtochukwu438/quizgen-llama3.git {REPO}
sys.path.append(REPO)
from src.generate import load_model, generate_quiz
from src.notes import load_notes
from src.rag import NotesIndex
OUT = "/content/drive/MyDrive/quizgen"

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.82 KiB | 1.41 MiB/s, done.
From https://github.com/Somtochukwu438/quizgen-llama3
   8664ec8..93be0fa  main       -> origin/main
Updating 8664ec8..93be0fa
Fast-forward
 app/app.py           | 11 +++++---
 src/answerability.py | 75 ++++++++++++++++++++++++++++++++++++++++++++++++++++
 src/generate.py      | 48 ++++++++++++++++++++++++---------
 3 files changed, 119 insertions(+), 15 deletions(-)
 create mode 100644 src/answerability.py


In [5]:
from datasets import load_dataset

model, tok = load_model(f"{REPO}/adapters")

test = load_dataset("allenai/sciq")["test"].filter(lambda x: x["support"].strip() != "")
eval_set = test.shuffle(seed=42).select(range(300))
with open(f"{OUT}/finetuned_outputs.jsonl") as f:
    saved = [json.loads(line) for line in f][:8]

recs = generate_quiz(model, tok, eval_set["support"][:8], max_tries=1)
matches = sum(r["raw"] == s["raw"] for r, s in zip(recs, saved))
print(f"{matches}/8 outputs identical to the saved evaluation run")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


7/8 outputs identical to the saved evaluation run


In [6]:
passages = load_notes(f"{REPO}/data/notes")
index = NotesIndex(passages)
print(len(passages), "passages indexed")
for topic in ["sleep stages", "how we feel pain", "brain imaging methods", "sense of smell"]:
    hits = index.search(topic, k=2)
    print(f"{topic!r} -> " + " | ".join(h["section"] for h in hits))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

12 passages indexed
'sleep stages' -> Sleep architecture | Functions and disorders
'how we feel pain' -> Body and pain | Organizing the nervous system
'brain imaging methods' -> Methods | Major brain systems
'sense of smell' -> Taste and smell | Body and pain


In [7]:
hits = index.search("sleep stages", k=3)
quiz = generate_quiz(model, tok, [h["text"] for h in hits])
for h, q in zip(hits, quiz):
    print(f"[{h['week'][:7]} · {h['section']}]  attempts: {q['attempts']}")
    if q["parsed"]:
        p = q["parsed"]
        print("Q:", p["question"], "| A:", p["answer"], "| D:", p["distractors"])
    else:
        print("FAILED:", q["error"])
    print("-" * 70)

[Week 9: · Sleep architecture]  attempts: 2
Q: What does nrem sleep progress through? | A: characteristic eeg patterns | D: ['characteristic eeg patterns and slower patterns', 'characteristic eeg patterns and slower waves', 'characteristic eeg patterns and slower activity']
----------------------------------------------------------------------
[Week 9: · Functions and disorders]  attempts: 1
Q: What supports learning, emotional regulation, metabolic health, immune function, and safety? | A: sleep | D: ['exercise', 'nutrition', 'medication']
----------------------------------------------------------------------
[Week 9: · Two-process regulation]  attempts: 1
Q: What is the name of the nucleus that coordinates daily rhythms using light input? | A: suprachiasmatic nucleus | D: ['hypothalamus', 'hypothalamus nucleus', 'hypothalamus gland']
----------------------------------------------------------------------


In [11]:
for i, (r, s) in enumerate(zip(recs, saved)):
    if r["raw"] != s["raw"]:
        print(f"Passage {i}")
        print("NOW:  ", r["raw"])
        print("SAVED:", s["raw"])

Passage 3
NOW:   {"question": "What is the protein in red blood cells which transports oxygen around the body?", "answer": "hemoglobin", "distractors": ["carbon", "nitrogen", "oxygen"]}
SAVED: {"question": "What is the protein in red blood cells which transports oxygen around the body?", "answer": "hemoglobin", "distractors": ["hematocrit", "hemoglobin", "hemoglobin"]}


In [12]:
!git -C {REPO} pull
import importlib, src.generate, app.app
importlib.reload(src.generate)
importlib.reload(app.app)
from src.answerability import AnswerChecker
from app.app import build_app

checker = AnswerChecker()
p = next(x["text"] for x in passages if x["section"] == "Sleep architecture")
q = "REM sleep includes cortical activation, vivid dreaming, rapid eye movements, and muscle atonia. What type of sleep does this occur in?"
print(checker.answer([q], [p]))
print(checker.answerable([q, q], ["nrem sleep", "rem sleep"], [p, p], threshold=0.67))

Already up to date.


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

['NREM']
[False, False]


In [14]:
tests = [
    ("Sleep architecture",
     "Which type of sleep includes cortical activation, vivid dreaming, rapid eye movements, and muscle atonia?",
     "rem sleep"),
    ("Two-process regulation",
     "What is the name of the nucleus that coordinates daily rhythms using light input?",
     "suprachiasmatic nucleus"),
    ("Functions and disorders",
     "What supports learning, emotional regulation, metabolic health, immune function, and safety?",
     "sleep"),
    ("Sleep architecture", q, "nrem sleep"),
]
ps = [next(x["text"] for x in passages if x["section"] == s) for s, _, _ in tests]
qs = [t[1] for t in tests]
golds = [t[2] for t in tests]
print(checker.answer(qs, ps))
print(checker.answerable(qs, golds, ps, threshold=0.67))

['REM', 'suprachiasmatic', 'Sleep', 'NREM']
[False, False, True, False]


In [ ]:
demo = build_app(model, tok, index)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://59cc34de219a3d1b54.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
